# M7 lab: what could happen? Scenarios, Monte Carlo and global sensitivity

*Energy Systems, Step by Step* · Modeler tier · lab for lesson M7 · about 45 minutes, plus 60 minutes of practice at the end.

You will rerun the lesson's Monte Carlo study in Python and check your numbers against the page's with `assert` cells, then use the standard tools:

1. The toy: a 100 MW load waits for its grid connection (case file CF7's illustrative numbers).
2. The page's own random-number generator and Latin hypercube, ported exactly: same seed, same runs.
3. 1,000 runs: percentiles, a chance above a budget, and the Monte Carlo error from 20 repeats.
4. Global sensitivity: Spearman's rank correlation with a tie rule, and what a narrower range does to it.
5. SciPy's Latin hypercube and Spearman's ρ.
6. Optional extension: variance shares (Sobol indices), beyond the lesson's outcomes.

**How to run it.** Put this notebook and `m07-monte-carlo.json` (the *Lesson data* button on the lesson page) in one folder. In Google Colab: *File → Upload notebook*, then upload the data file with the folder icon on the left, then *Runtime → Run all*. Locally: Python 3.9 or later with `numpy` and `scipy` 1.9 or later; `matplotlib` draws the histograms if you have it. Everything runs in a few seconds.

**The data.** Illustrative round numbers in one price level (no inflation), invented ranges around CF7's values. Every chance below is a share of model runs: a statement about the model under its assumed inputs, not a forecast of any real project.

In [ ]:
import itertools
import json
from pathlib import Path

import numpy as np
import scipy
from scipy import stats
from scipy.stats import qmc

major, minor = (int(x) for x in scipy.__version__.split(".")[:2])
assert (major, minor) >= (1, 9), "this lab needs SciPy 1.9 or later (pip install -U scipy)"
assert Path("m07-monte-carlo.json").exists(), "m07-monte-carlo.json not found: put it next to this notebook (in Colab, upload it with the folder icon)."
D = json.load(open("m07-monte-carlo.json"))
REF = D["reference"]
try:
    import matplotlib.pyplot as plt
    HAVE_PLT = True
except ImportError:
    HAVE_PLT = False
print("NumPy", np.__version__, "· SciPy", scipy.__version__, "· matplotlib", "yes" if HAVE_PLT else "no (text histograms)")

def show(q, v):
    # an input's value in its unit: the discount rate in %, the never-arrives chance in percentage points (the model
    # itself works with fractions: 0.005 a month)
    return f"{100 * v:.0f}%" if q["key"] == "rate" else f"{100 * v:.2f}" if q["key"] == "strand" else f"{v:,.0f}"

for q in D["inputs"]:
    print(f"{q['label']:28s} {show(q, q['lo']):>12s} to {show(q, q['hi']):<12s} ({q['unit']}); CF7's value {show(q, q['base'])}")

## 1. The toy

A load of $P$ = 100 MW waits $\tau$ months. Each run discounts at its own yearly rate $r$, month by month: $i = (1+r)^{1/12} - 1$ and $A(\tau) = \frac{1 - (1+i)^{-\tau}}{i}$, the present value of \$1 a month for $\tau$ months.

- **The developer:** lost margin $m$ per MW a month and the carrying cost of idle capital $K$: $D = (mP + Ki)\,A(\tau)$.
- **Other customers:** the carrying cost of the upgrades $U$, and their expected stranding at the end of the wait, with the chance $h\tau$ (at most 50%) that the load never arrives: $R = U i A(\tau) + \min(0.5,\ h\tau)\,U (1+i)^{-\tau}$.
- **The never-arrives chance** is added each month, not compounded: 0.5 percentage points a month gives 12% after 24 months.
- **Its unit in the formula:** $h$ is a fraction, $h$ = 0.005 a month (0.5 points), so $h\tau$ = 0.12 and the cap is 0.5. The data file stores it that way.

The two accounts stay separate, as in CF7. The functions take NumPy arrays, so one call runs every sample.

In [ ]:
T = D["toy"]
KEYS = [q["key"] for q in D["inputs"]]
BASE = {q["key"]: q["base"] for q in D["inputs"]}

def outputs(x, tau):
    # x: a dict of arrays (or numbers), one per input; returns the developer's and other customers' present values in $
    i = (1 + np.asarray(x["rate"], float)) ** (1 / 12) - 1
    A = (1 - (1 + i) ** -tau) / i
    dev = (x["margin"] * T["loadMW"] + x["capital"] * i) * A
    rate = x["upgrade"] * i * A + np.minimum(T["strandCap"], x["strand"] * tau) * x["upgrade"] * (1 + i) ** -tau
    return dev, rate

for tau in D["waits"]:
    dev, rate = outputs(BASE, tau)
    print(f"{tau} months at CF7's values: developer ${dev / 1e6:.1f}M, other customers ${rate / 1e6:.1f}M")
    ref = REF["base"][str(tau)]
    assert abs(dev / ref["dev"] - 1) < 1e-9 and abs(rate / ref["rate"] - 1) < 1e-9

## 2. The page's generator and its Latin hypercube

A study is reproducible when its seed is stated and its sampler is written down. The page uses **mulberry32**, a small 32-bit generator; the cell below ports it, with the 32-bit arithmetic made explicit. (A generator like this is fine for teaching; for your own work use NumPy's `default_rng`.)

**Latin hypercube, as the page does it.** For each input in order (margin, capital, discount rate, upgrades, never-arrives chance): shuffle the slice numbers $0, \dots, N-1$ (Fisher–Yates, from the last position down), then give run $a$ the draw $u = (\text{slice}_a + \text{one more random number}) / N$. The value is $\text{lo} + u\,(\text{hi} - \text{lo})$. Every input then has exactly one draw in each of its $N$ equal slices.

In [ ]:
MASK = 0xFFFFFFFF

def imul(a, b):
    # JavaScript's Math.imul: the low 32 bits of a * b, read as an unsigned number here
    return (a * b) & MASK

def mulberry32(seed):
    a = seed & MASK
    def draw():
        nonlocal a
        a = (a + 0x6D2B79F5) & MASK
        t = imul(a ^ (a >> 15), a | 1)
        t ^= (t + imul(t ^ (t >> 7), t | 61)) & MASK
        return ((t ^ (t >> 14)) & MASK) / 4294967296
    return draw

def units(n, k, seed, method="lhs"):
    # unit draws U[j][a] in [0, 1) for input j and run a, in the page's order of draws; also the slices (None for random)
    r = mulberry32(seed)
    if method == "random":
        U = np.empty((k, n))
        for a in range(n):
            for j in range(k):
                U[j, a] = r()
        return U, None
    U, P = np.empty((k, n)), np.empty((k, n), dtype=int)
    for j in range(k):
        p = list(range(n))
        for a in range(n - 1, 0, -1):
            b = int(r() * (a + 1))
            p[a], p[b] = p[b], p[a]
        for a in range(n):
            U[j, a] = (p[a] + r()) / n
        P[j] = p
    return U, P

def study(n=D["runs"], seed=D["seed"], tau=D["wait"], method="lhs", inputs=D["inputs"]):
    U, P = units(n, len(inputs), seed, method)
    X = {q["key"]: q["lo"] + U[j] * (q["hi"] - q["lo"]) for j, q in enumerate(inputs)}
    dev, rate = outputs(X, tau)
    return {"X": X, "U": U, "P": P, "dev": dev, "rate": rate}

S = study()
for a, ref in enumerate(REF["firstRuns"]):
    for k in KEYS:
        assert abs(S["X"][k][a] / ref[k] - 1) < 1e-12, (a, k)
for j, k in enumerate(KEYS):
    assert sorted(np.floor(S["U"][j] * D["runs"]).astype(int)) == list(range(D["runs"])), k
print("Seed", D["seed"], "· run 1:", {k: round(float(S["X"][k][0]), 6) for k in KEYS})
print("Every input has exactly one draw in each of its", D["runs"], "slices; the first three runs equal the page's.")

Ten runs make the difference visible. The page's step 5 counts the empty slices of the margin and the discount rate for seed 7: try other seeds.

In [ ]:
def empty_slices(U, j, n):
    return n - len(set(np.floor(U[j] * n).astype(int)))

n = D["preview"]
for method in ("random", "lhs"):
    U, _ = units(n, len(KEYS), D["seed"], method)
    e = (empty_slices(U, 0, n), empty_slices(U, 2, n))
    print(f"{method:6s} seed {D['seed']}: {e[0]} of {n} margin slices and {e[1]} of {n} discount-rate slices empty")
    assert e == (REF["preview"][method]["margin"], REF["preview"][method]["rate"])

## 3. 1,000 runs: read the distribution, not one number

Percentiles by linear interpolation at rank $(N-1)p$ (NumPy's default, the page's rule since K5). The page shows the middle 80% of runs, from P10 to P90.

In [ ]:
def text_hist(v, lo, hi, w):
    edges = np.arange(lo, hi + w / 2, w)
    counts, _ = np.histogram(v, bins=edges)
    for c, e in zip(counts, edges):
        if c:
            print(f"  ${e / 1e6:5.0f}M  {'#' * max(1, int(round(c / 4)))} {c}")

dev = S["dev"]
p10, p50, p90 = np.quantile(dev, [0.1, 0.5, 0.9])
print(f"median ${p50 / 1e6:.1f}M; 80% of runs from ${p10 / 1e6:.1f}M to ${p90 / 1e6:.1f}M; runs ${dev.min() / 1e6:.1f}M to ${dev.max() / 1e6:.1f}M")
for k, v in zip(("p10", "p50", "p90", "min", "max"), (p10, p50, p90, dev.min(), dev.max())):
    assert abs(v / REF["main"]["dev"][k] - 1) < 1e-9, k
lo_c, hi_c = REF["corners"]["dev"]
corners = []
for ends in itertools.product((0, 1), repeat=len(KEYS)):
    x = {q["key"]: (q["hi"] if e else q["lo"]) for q, e in zip(D["inputs"], ends)}
    corners.append(outputs(x, D["wait"])[0])
assert abs(min(corners) / lo_c - 1) < 1e-9 and abs(max(corners) / hi_c - 1) < 1e-9
print(f"corners of the five ranges: ${min(corners) / 1e6:.1f}M and ${max(corners) / 1e6:.1f}M")
# An exact corner has zero chance with continuous inputs, so count the runs NEAR one: the three inputs of the developer's
# cost (margin, capital, rate) all in the same outer tenth of their ranges. Its chance is 0.1 ** 3 per corner.
f, Q = D["near"], {q["key"]: q for q in D["inputs"]}
u = np.array([(S["X"][k] - Q[k]["lo"]) / (Q[k]["hi"] - Q[k]["lo"]) for k in ("margin", "capital", "rate")])
near_hi, near_lo = int((u >= 1 - f).all(axis=0).sum()), int((u < f).all(axis=0).sum())
print(f"near the high-cost corner: {near_hi} runs; near the low-cost one: {near_lo}; a chance of {f} ** 3 per corner predicts {2 * f ** 3 * D['runs']:.0f}")
assert (near_lo, near_hi) == (REF["near"]["lo"], REF["near"]["hi"])
if HAVE_PLT:
    plt.figure(figsize=(7, 3))
    plt.hist(dev / 1e6, bins=np.arange(100, 255, 5), color="#3b6fb6", alpha=0.8)
    plt.axvspan(p10 / 1e6, p90 / 1e6, color="#3b6fb6", alpha=0.12, label="P10 to P90")
    for c in (lo_c, hi_c):
        plt.axvline(c / 1e6, color="#b33", ls="--")
    plt.xlabel("developer's cost, present value ($M)"); plt.ylabel("runs"); plt.legend(); plt.show()
else:
    text_hist(dev, 100e6, 250e6, 5e6)

**A chance under the model's inputs.** The share of runs above a budget, for each wait scenario. Each run's inputs are the same across the three scenarios (same seed): only the wait changes.

In [ ]:
BUDGET = D["budget"]["start"] * 1e6
for tau in D["waits"]:
    d = study(tau=tau)["dev"]
    n_above = int((d > BUDGET).sum())
    print(f"{tau} months: {n_above} of {D['runs']:,} runs above ${BUDGET / 1e6:.0f}M ({n_above / D['runs']:.1%})")
    assert n_above == REF["budget"][str(tau)]["n"]

## 4. How sure is a Monte Carlo number?

Repeat the whole run with seeds 1 to 20 (seed 7 is the run above). The standard deviation of the 20 estimates is the **standard error** of one estimate: the Monte Carlo error. Twenty repeats make that error itself rough (about ± 16%).

In [ ]:
SEEDS = range(D["repeats"]["first"], D["repeats"]["last"] + 1)

def repeat(n, method="lhs", tau=D["wait"]):
    out = []
    for s in SEEDS:
        d = study(n=n, seed=s, tau=tau, method=method)["dev"]
        out.append({"p90": np.quantile(d, 0.9), "mean": d.mean(), "share": (d > BUDGET).mean()})
    return out

se = lambda reps, k: np.std([r[k] for r in reps], ddof=1)
R = {(n, m): repeat(n, m) for n in D["repeats"]["sizes"] for m in ("lhs", "random")}
for (n, m), reps in R.items():
    print(f"{n:5,d} runs, {m:6s}: P90 ± ${2 * se(reps, 'p90') / 1e6:.1f}M, mean ± ${2 * se(reps, 'mean') / 1e6:.2f}M, share above $200M ± {2 * se(reps, 'share'):.1%} (2 standard errors)")
assert abs(se(R[(100, "lhs")], "p90") / REF["p90se"]["100"] - 1) < 1e-4
assert abs(se(R[(1000, "lhs")], "p90") / REF["p90se"]["1000"] - 1) < 1e-4
assert abs(se(R[(1000, "random")], "p90") / REF["p90se"]["1000random"] - 1) < 1e-4
assert abs(se(R[(1000, "lhs")], "mean") / REF["meanse"]["lhs"] - 1) < 1e-4
assert abs(se(R[(1000, "random")], "mean") / REF["meanse"]["random"] - 1) < 1e-4
assert abs(se(R[(1000, "lhs")], "share") / REF["budget"]["24"]["se"] - 1) < 1e-4
p = REF["budget"]["24"]["n"] / D["runs"]
print(f"K5's formula for random draws, sqrt(p(1 - p)/N): {np.sqrt(p * (1 - p) / D['runs']):.2%} for the share; the hypercube's repeats: {se(R[(1000, 'lhs')], 'share'):.2%}")

## 5. Which inputs matter, all varying together?

**Spearman's rank correlation** ρ: the correlation of the ranks of an input and of the cost. With a Latin hypercube, an input's ranks are its slice numbers plus one, so no sort is needed. The cell checks your own ranks against `scipy.stats.spearmanr`.

**The tie rule**, stated before looking:

- Order the inputs by |ρ|.
- One input ranks *clearly above* another when their gap is more than two standard errors of that gap (the standard error again from the 20 repeats). Every pair is tested.
- An input's rank is then a range: from 1 + the inputs clearly above it to the number of inputs not clearly below it. "2–3" means tied: not clearly ordered against its neighbors.

In [ ]:
def rhos(st, out):
    ro = stats.rankdata(st[out])
    return {k: np.corrcoef(st["P"][j] + 1 if st["P"] is not None else stats.rankdata(st["X"][k]), ro)[0, 1] for j, k in enumerate(KEYS)}

def rank_ranges(rh, reps_rho):
    # each input's rank as (best, worst): 1 + the inputs clearly above it, and the number not clearly below it
    order = sorted(KEYS, key=lambda k: -abs(rh[k]))   # Python's sort is stable, as the page's
    def clear(a, b):                                   # a clearly above b
        gap = abs(rh[a]) - abs(rh[b])
        return gap > 2 * np.std([abs(r[a]) - abs(r[b]) for r in reps_rho], ddof=1)
    n = len(order)
    return {x: (1 + sum(clear(y, x) for y in order[:i]), n - sum(clear(x, y) for y in order[i + 1:])) for i, x in enumerate(order)}

def show_ranks(rh, R):
    lab = lambda r: str(r[0]) if r[0] == r[1] else f"{r[0]}-{r[1]}"
    return ", ".join(f"{k} {rh[k]:.2f} (rank {lab(R[k])})" for k in sorted(R, key=lambda k: -abs(rh[k])))

reps = [study(seed=s) for s in SEEDS]
for out, name, key in (("dev", "the developer", "rhoDev"), ("rate", "other customers", "rhoRate")):
    rh = rhos(S, out)
    for k in KEYS:
        assert abs(rh[k] - REF["main"][key][k]) < 1e-9, (out, k)
        assert abs(rh[k] - stats.spearmanr(S["X"][k], S[out])[0]) < 1e-12
    R = rank_ranges(rh, [rhos(r, out) for r in reps])
    print(f"{name}: " + show_ranks(rh, R))
    top = "margin" if out == "dev" else "upgrade"
    assert R[top] == (1, 1), "the page's step 9: " + top + " ranks first alone"
    if out == "dev":
        assert R["capital"] == R["rate"] == (2, 3), "the page's step 9: capital and the discount rate tied, rank 2-3"

**Ranges set the ranking.** Narrow the margin's range to \$45,000 ± \$6,000 and rank again (the page's step 10). ρ ranks inputs within the ranges you assumed, not in the world.

In [ ]:
def narrowed(half):
    return [dict(q, lo=q["base"] - half, hi=q["base"] + half) if q["key"] == "margin" else q for q in D["inputs"]]

half = REF["flip"]["half"]
st = study(inputs=narrowed(half))
rh = rhos(st, "dev")
for k in KEYS:
    assert abs(rh[k] - REF["flip"]["rho"][k]) < 1e-9, k
R = rank_ranges(rh, [rhos(study(seed=s, inputs=narrowed(half)), "dev") for s in SEEDS])
print(f"margin ± ${half:,}: " + show_ranks(rh, R))
assert R["margin"][0] > 1, "another input ranks clearly above the margin: it is no longer even tied first"

# one step wider, at ± $7,000: the margin and capital are within two standard errors, and so are capital and the
# discount rate, but the margin ranks clearly above the discount rate; ranges keep every pair's verdict
st7 = study(inputs=narrowed(7000)); rh7 = rhos(st7, "dev")
R7 = rank_ranges(rh7, [rhos(study(seed=s, inputs=narrowed(7000)), "dev") for s in SEEDS])
print("margin ± $7,000: " + show_ranks(rh7, R7))
assert (R7["margin"], R7["capital"], R7["rate"]) == ((1, 2), (1, 3), (2, 3))
print("The lab matches the browser.")

## 6. The standard tools: SciPy's Latin hypercube and Spearman's ρ

From here on, use a well-tested generator and sampler. Your numbers will not equal the page's; they should be statistically consistent with them.

- A random difference can exceed the error bars: two independent estimates differ by more than two standard errors of their difference about 1 time in 20.
- To compare two estimates, use the standard error of their difference: for two independent estimates with the same standard error, √2 times it.
- The check below stops only at five standard errors of the difference. It looks for mistakes; it is not a guarantee.

In [ ]:
def scipy_lhs(n, seed):
    try:
        sampler = qmc.LatinHypercube(d=len(KEYS), rng=np.random.default_rng(seed))
    except TypeError:                      # SciPy before 1.15 calls the argument seed
        sampler = qmc.LatinHypercube(d=len(KEYS), seed=np.random.default_rng(seed))
    u = sampler.random(n)                  # n x k in [0, 1): one draw in each of n slices of every column
    lo = np.array([q["lo"] for q in D["inputs"]]); hi = np.array([q["hi"] for q in D["inputs"]])
    return {k: col for k, col in zip(KEYS, (lo + u * (hi - lo)).T)}

X = scipy_lhs(D["runs"], 2026)
dev, rate = outputs(X, D["wait"])
p90 = np.quantile(dev, 0.9)
se_diff = np.sqrt(2) * REF["p90se"]["1000"]      # two independent 1,000-run hypercube estimates, each with the page's standard error
gap = abs(p90 - REF["main"]["dev"]["p90"]) / se_diff
print(f"SciPy's hypercube, seed 2026: P90 ${p90 / 1e6:.1f}M; the page's ${REF['main']['dev']['p90'] / 1e6:.1f}M; apart by {gap:.1f} standard errors of the difference")
for k in KEYS:
    print(f"  {k:8s} rho with the developer {stats.spearmanr(X[k], dev)[0]:+.2f}, with other customers {stats.spearmanr(X[k], rate)[0]:+.2f}")
assert gap < 5, "five standard errors of the difference apart: a sign of a mistake, not a proof (check the ranges and the toy)"

## 7. Optional extension: variance shares (Sobol indices)

This section goes beyond the lesson's outcomes; skip it on a first pass. A rank correlation reads monotone effects only. **Sobol indices** split the variance of the cost among the inputs:

- the first-order index $S_i$ is the share of the variance that input $i$ explains on its own;
- the total index $S_{Ti}$ adds every interaction it takes part in, so $S_{Ti} - S_i$ measures its interactions.

**Construction.** Draw two independent matrices $A$ and $B$ of $N$ rows, one column per input. $A_B^{(i)}$ is $A$ with column $i$ taken from $B$. That makes $N(k + 2)$ runs: 57,344 here with $N$ = 8,192, a fraction of a second for this toy.

**Estimation** (Saltelli and co-authors, 2010). With $V$ the variance of all the outputs:

- $S_i \approx \frac{1}{N}\sum f(B)\,[f(A_B^{(i)}) - f(A)] / V$;
- $S_{Ti} \approx \frac{1}{2N}\sum [f(A) - f(A_B^{(i)})]^2 / V$.

Subtract the mean of the outputs first: the estimates' expected values stay the same and their noise drops.

**Uncertainty: a bootstrap.** Draw $N$ row numbers at random *with replacement* (some rows twice, some not at all) and estimate every index again from those rows. Repeat 200 times: the standard deviation of the 200 estimates is each index's standard error.

In [ ]:
def sobol(out_index, N=8192, seed=2026, boot=200):
    g = np.random.default_rng(seed)
    lo = np.array([q["lo"] for q in D["inputs"]]); hi = np.array([q["hi"] for q in D["inputs"]])
    A = lo + g.random((N, len(KEYS))) * (hi - lo); B = lo + g.random((N, len(KEYS))) * (hi - lo)
    f = lambda Z: outputs({k: Z[:, j] for j, k in enumerate(KEYS)}, D["wait"])[out_index]
    fA, fB = f(A), f(B)
    fAB = []
    for j in range(len(KEYS)):
        Z = A.copy(); Z[:, j] = B[:, j]; fAB.append(f(Z))
    m = np.mean(np.concatenate([fA, fB]))                             # center the outputs: same estimates, far less noise
    fA, fB, fAB = fA - m, fB - m, [x - m for x in fAB]
    def est(rows):
        V = np.var(np.concatenate([fA[rows], fB[rows]]))
        return [(np.mean(fB[rows] * (fAB[j][rows] - fA[rows])) / V, np.mean((fA[rows] - fAB[j][rows]) ** 2) / (2 * V)) for j in range(len(KEYS))]
    point = est(np.arange(N))
    bs = np.array([est(g.integers(0, N, N)) for _ in range(boot)])     # boot x k x 2
    return point, bs.std(axis=0, ddof=1)

for out_index, name, key in ((0, "the developer", "rhoDev"), (1, "other customers", "rhoRate")):
    point, err = sobol(out_index)
    print(f"{name}:  input     S_i (± 2 SE)      S_Ti (± 2 SE)     rho^2 from section 5")
    for j, k in enumerate(KEYS):
        print(f"  {k:8s}  {point[j][0]:5.2f} ± {2 * err[j][0]:.2f}     {point[j][1]:5.2f} ± {2 * err[j][1]:.2f}     {REF['main'][key][k] ** 2:5.2f}")
    s_sum = sum(p[0] for p in point)
    print(f"  sum of first-order indices {s_sum:.2f}: close to 1 means the inputs act almost additively")
    assert all(p[1] >= p[0] - 2 * (e[0] + e[1]) for p, e in zip(point, err)) and abs(s_sum - 1) < 0.1, "the toy acts almost additively"

**Read it.** For this toy the first-order indices add up to almost 1 and each $S_{Ti}$ is close to its $S_i$: the inputs barely interact, so the rank correlations and the variance shares tell the same story. When the two disagree, trust neither alone: look at the scatter of the cost against the input.

## 8. Practice (about 60 minutes)

Write your own cells below each task. Predict first, then run, and report each number with its run count, seed and Monte Carlo error.

1. **The wait as an input** (about 15 minutes). Make the wait uniform on 12 to 36 months instead of a scenario (add it as a sixth input of the hypercube). Which input ranks first now, and what does that say about treating the wait as a scenario?
2. **Another shape** (about 10 minutes). Give the margin a triangular distribution on the same range, peaked at \$45,000 (`scipy.stats.triang`, applied to the hypercube's unit draws). How do the P90 and the margin's ρ change, and why?
3. **Inputs that move together** (about 15 minutes). Suppose a project whose upgrades cost more is also more likely never to arrive.
    - The helper in the next cell induces a rank correlation near 0.6 between those two inputs: a simplified, two-input version of Iman and Conover's method.
    - It keeps the upgrades column in place and reorders the never-arrives column. Both keep their sampled values, and the other inputs stay with their runs.
    - 0.6 is an approximate target: measure the rank correlation you achieved and accept 0.55 to 0.65.
    - Then compare other customers' P90 with and without the link, each with its Monte Carlo error.

In [ ]:
def induce_rank_correlation(keep, move, target, seed):
    # a simplified, two-input version of Iman and Conover's method: return `move` reordered so that its ranks follow `keep`'s with a rank
    # correlation near `target`. `keep` stays in place, and so do the other inputs; `move` keeps its values (its sample),
    # only their order changes.
    g = np.random.default_rng(seed)
    n = len(keep)
    z_keep = stats.norm.ppf(stats.rankdata(keep) / (n + 1))          # normal scores in the order of keep's ranks
    r = 2 * np.sin(np.pi * target / 6)                                # the normal correlation whose rank correlation is target
    z_move = r * z_keep + np.sqrt(1 - r ** 2) * g.standard_normal(n)  # a correlated partner for each run
    place = stats.rankdata(z_move, method="ordinal").astype(int) - 1  # 0 .. n - 1: the rank each run's value takes
    return np.sort(move)[place]

up, strand = S["X"]["upgrade"], S["X"]["strand"]
linked = induce_rank_correlation(up, strand, 0.6, seed=2026)
achieved = stats.spearmanr(up, linked)[0]
assert np.array_equal(np.sort(linked), np.sort(strand)), "the never-arrives column keeps its sampled values"
print(f"rank correlation of the upgrades and the never-arrives chance: before {stats.spearmanr(up, strand)[0]:+.3f}, target 0.6, achieved {achieved:.3f}")
assert abs(achieved - 0.6) <= 0.05, "0.6 is an approximate target: expect 0.55 to 0.65"
# your turn: put `linked` in place of the never-arrives column and compare other customers' P90 with and without the link

4. **The open code** (about 10 minutes; a separate environment). This task runs outside the notebook: clone the public `code-delayed-energization` repository and install it as its README's quick start says, with Python 3.10 or later. Then run `code-delay example my_scenario.json --grid grid.json --ranges ranges.json`, then `code-delay lhs my_scenario.json ranges.json --samples 500 --seed 1 --out-dir outputs/lhs` and again with `--seed 2`. Which rank correlations in `rank_correlations.csv` move by more than two of your standard errors?
5. **Write it up** (about 10 minutes). In five sentences, report your Monte Carlo study as a modeler would: the scenario, the inputs with their ranges and shapes, the sampler with its seed and run count, the P90 with its Monte Carlo error, and the ranking with the tie rule; then say what the runs cannot show.